# SentinelML SHAP Explainability

SHAP explanations for the tuned XGBoost model from Experiment 03.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.explain import (
    compare_shap_to_eda_correlations,
    compute_shap_values,
    explain_single_prediction,
    get_global_feature_importance,
)
from src.train_models import _build_xgboost, _compute_scale_pos_weight, load_processed_splits

## Train Tuned XGBoost

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = load_processed_splits(
    PROJECT_ROOT / "data" / "processed"
)

best_params = {
    "max_depth": 6,
    "learning_rate": 0.04087659627832751,
    "n_estimators": 600,
    "subsample": 0.8654555403181324,
    "colsample_bytree": 0.9958149399088472,
    "min_child_weight": 8,
}
scale_pos_weight = _compute_scale_pos_weight(y_train)
model = _build_xgboost(scale_pos_weight, **best_params)
model.fit(X_train, y_train)
y_val_proba = model.predict_proba(X_val)[:, 1]
y_val_pred = y_val_proba >= 0.5

## Global SHAP Importance

In [ ]:
shap_values = compute_shap_values(model, X_val)
importance = get_global_feature_importance(shap_values, X_val.columns)
correlation_check = compare_shap_to_eda_correlations(importance)
importance.head(15)

In [ ]:
shap.summary_plot(shap_values, X_val, max_display=15, show=True)

## Individual Waterfall Explanations

In [ ]:
tp_idx = X_val.index[(y_val == 1) & y_val_pred][0]
fp_idx = X_val.index[(y_val == 0) & y_val_pred][0]
fn_idx = X_val.index[(y_val == 1) & ~y_val_pred][0]

explainer = shap.TreeExplainer(model)
case_indices = {
    "True positive fraud catch": tp_idx,
    "False positive normal flagged as fraud": fp_idx,
    "False negative missed fraud": fn_idx,
}

for label, idx in case_indices.items():
    row = X_val.loc[[idx]]
    row_shap = explainer.shap_values(row)
    expected_value = explainer.expected_value
    explanation = shap.Explanation(
        values=row_shap[0],
        base_values=expected_value,
        data=row.iloc[0],
        feature_names=X_val.columns,
    )
    print(f"{label}: index={idx}, y={int(y_val.loc[idx])}, p_fraud={float(y_val_proba[X_val.index.get_loc(idx)]):.6f}")
    display(explain_single_prediction(model, explainer, row, X_val.columns)["breakdown"].head(10))
    shap.plots.waterfall(explanation, max_display=12, show=True)